<a href="https://colab.research.google.com/github/Atharv2200/GroupDNA-WhatsApp-Analytics/blob/main/GroupDna_Atharv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 GroupDNA: WhatsApp Group Chat Analyzer

Atharv Mandekar

In [ ]:
import os
from datetime import datetime, timedelta
import numpy as np

def get_count(item):
    return item[1]

FILE_PATH = "hostel_bois.txt"

def is_new_message(line):
    if len(line) > 8:
        if line[2] == '/' and line[5] == '/':
            return True
    return False

with open(FILE_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    lines = f.readlines()

messages = []
system_count = 0
media_count = 0
deleted_count = 0
current_msg = None

for raw_line in lines:
    line = raw_line.strip()
    if line == "":
        continue

    if is_new_message(line):
        if current_msg != None:
            messages.append(current_msg)

        parts = line.split(" - ", 1)

        if len(parts) < 2:
            parts = line.split(" ", 2)
            if len(parts) >= 3 and "," in parts[0]:
                ts_part = parts[0] + " " + parts[1]
                ts_part = ts_part.strip()
                rest = parts[2]
            else:
                system_count = system_count + 1
                current_msg = None
                continue
        else:
            ts_part = parts[0]
            rest = parts[1]

        if ":" in rest:
            sender_and_content = rest.split(":", 1)
            sender = sender_and_content[0].strip()
            content = sender_and_content[1].strip()

            is_media = False
            if "<Media omitted>" in content:
                is_media = True
                media_count = media_count + 1

            is_deleted = False
            if "This message was deleted" in content or "You deleted this message" in content:
                is_deleted = True
                deleted_count = deleted_count + 1

            current_msg = {
                "timestamp_str": ts_part,
                "sender": sender,
                "text": content,
                "is_media": is_media,
                "is_deleted": is_deleted
            }
        else:
            system_count = system_count + 1
            current_msg = None
    else:
        if current_msg != None:
            current_msg["text"] = current_msg["text"] + " " + line

if current_msg != None:
    messages.append(current_msg)

In [ ]:
def parse_date(ts_str):
    ts_str = ts_str.replace('\u202f', ' ').strip()
    formats = ["%d/%m/%y, %H:%M", "%d/%m/%Y, %H:%M", "%d/%m/%y, %I:%M %p", "%d/%m/%Y, %I:%M %p"]

    for fmt in formats:
        try:
            return datetime.strptime(ts_str, fmt)
        except ValueError:
            continue
    return None

person_counts = {}
day_counts = {}
hour_counts = {}
valid_dates = []

for msg in messages:
    sender = msg["sender"]

    if sender in person_counts:
        person_counts[sender] = person_counts[sender] + 1
    else:
        person_counts[sender] = 1

    dt = parse_date(msg["timestamp_str"])
    if dt != None:
        msg["dt"] = dt
        valid_dates.append(dt)

        day_str = dt.strftime("%d %B %Y")
        if day_str in day_counts:
            day_counts[day_str] = day_counts[day_str] + 1
        else:
            day_counts[day_str] = 1

        hour_str = dt.strftime("%H:00")
        if hour_str in hour_counts:
            hour_counts[hour_str] = hour_counts[hour_str] + 1
        else:
            hour_counts[hour_str] = 1

valid_dates.sort()
start_date = valid_dates[0]
end_date = valid_dates[-1]
total_days = (end_date.date() - start_date.date()).days + 1
if total_days < 1:
    total_days = 1

sorted_participants = sorted(person_counts.items(), key=get_count, reverse=True)

busiest_day = ("N/A", 0)
if len(day_counts) > 0:
    busiest_day = max(day_counts.items(), key=get_count)

busiest_hour = ("N/A", 0)
if len(hour_counts) > 0:
    busiest_hour = max(hour_counts.items(), key=get_count)

In [ ]:
STOP_WORDS = ['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'that', 'this', 'you', 'me', 'my', 'we', 'us', 'hai', 'ki', 'ko', 'se', 'ke', 'ka', 'ye', 'wo', 'to']

word_freq = {}
punctuations = "!()-[]{};:'\"\,<>./?@#$%^&*_~"

for msg in messages:
    if msg["is_media"] == True or msg["is_deleted"] == True:
        continue

    clean_text = ""
    for char in msg["text"].lower():
        if char not in punctuations:
            clean_text = clean_text + char
        else:
            clean_text = clean_text + " "

    words = clean_text.split()
    for w in words:
        if w != "" and w not in STOP_WORDS and len(w) > 2:
            if w in word_freq:
                word_freq[w] = word_freq[w] + 1
            else:
                word_freq[w] = 1

top_words = sorted(word_freq.items(), key=get_count, reverse=True)[:10]

In [ ]:
participant_names = []
for p in sorted_participants:
    participant_names.append(p[0])

activity_matrix = np.zeros((len(participant_names), 24), dtype=int)

for msg in messages:
    if "dt" in msg:
        p_idx = participant_names.index(msg["sender"])
        hour = msg["dt"].hour
        activity_matrix[p_idx, hour] = activity_matrix[p_idx, hour] + 1



for i in range(len(participant_names)):
    name = participant_names[i]
    row_max = activity_matrix[i].max()

    display_name = name[:8] if len(name) > 8 else name
    row_str = f"{display_name:<8} "

    for h in range(0, 24, 3):
        window_sum = activity_matrix[i, h:h+3].sum()

        char = '.'
        if row_max > 0 and window_sum > 0:
            pct = window_sum / (row_max * 3)
            if pct < 0.25:
                char = '.'
            elif pct < 0.50:
                char = '░'
            elif pct < 0.75:
                char = '▒'
            else:
                char = '█'

        row_str = row_str + f" {char}  "
    print(row_str)

In [ ]:
response_times = {}
active_dates = {}

for name in participant_names:
    response_times[name] = []
    active_dates[name] = []

last_sender = None
last_time = None

for msg in messages:
    if "dt" not in msg:
        continue

    sender = msg["sender"]
    dt = msg["dt"]

    if last_sender != None and last_sender != sender:
        gap = (dt - last_time).total_seconds()
        if gap > 0 and gap <= 43200:
            response_times[sender].append(gap)

    last_sender = sender
    last_time = dt

    date_only = dt.date()
    if date_only not in active_dates[sender]:
        active_dates[sender].append(date_only)

avg_responses = {}
for person in response_times:
    gaps = response_times[person]
    if len(gaps) > 0:
        avg_responses[person] = sum(gaps) / len(gaps)
    else:
        avg_responses[person] = 999999

streaks = {}
for person in participant_names:
    max_streak = 0
    curr_streak = 0

    for i in range(total_days):
        check_date = start_date.date() + timedelta(days=i)
        if check_date not in active_dates[person]:
            curr_streak = curr_streak + 1
            if curr_streak > max_streak:
                max_streak = curr_streak
        else:
            curr_streak = 0

    streaks[person] = max_streak



In [ ]:
archetypes = {}
caring_words = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder']
philosophy_words = ['life', 'time', 'meaning', 'future', 'career', 'destiny', 'bro']

for idx in range(len(participant_names)):
    person = participant_names[idx]

    p_msgs = []
    for m in messages:
        if m["sender"] == person:
            p_msgs.append(m)

    total_p_msgs = len(p_msgs)
    if total_p_msgs == 0:
        total_p_msgs = 1

    bursts = []
    curr_burst = 0
    for m in messages:
        if m["sender"] == person:
            curr_burst = curr_burst + 1
        else:
            if curr_burst > 0:
                bursts.append(curr_burst)
                curr_burst = 0
    if curr_burst > 0:
        bursts.append(curr_burst)

    avg_burst = 0
    if len(bursts) > 0:
        avg_burst = sum(bursts) / len(bursts)

    night_msgs = activity_matrix[idx, 23]
    for hour in range(5):
        night_msgs = night_msgs + activity_matrix[idx, hour]

    total_matrix_msgs = np.sum(activity_matrix[idx])
    if total_matrix_msgs == 0:
        total_matrix_msgs = 1
    night_pct = (night_msgs / total_matrix_msgs) * 100

    total_words = 0
    valid_msg_count = 0
    for m in p_msgs:
        if m["is_media"] == False:
            words_list = m["text"].split()
            total_words = total_words + len(words_list)
            valid_msg_count = valid_msg_count + 1

    avg_words = 0
    if valid_msg_count > 0:
        avg_words = total_words / valid_msg_count

    drama_count = 0
    for m in p_msgs:
        text = m["text"]
        if (len(text) > 3 and text.isupper()) or text.count('!') >= 2:
            drama_count = drama_count + 1
    drama_pct = (drama_count / total_p_msgs) * 100

    caring_score = 0
    phil_score = 0
    for m in p_msgs:
        text_lower = m["text"].lower()
        for w in caring_words:
            caring_score = caring_score + text_lower.count(w)
        for w in philosophy_words:
            phil_score = phil_score + text_lower.count(w)

    silent_pct = (streaks[person] / total_days) * 100

    if avg_burst >= 3.5:
        archetypes[person] = ("THE SPAMMER", f"avg {avg_burst:.1f} msgs in a row")
    elif night_pct >= 50:
        archetypes[person] = ("THE NIGHT OWL", f"{night_pct:.1f}% msgs at night")
    elif caring_score >= 8:
        archetypes[person] = ("THE GROUP MOM", f"caring score: {caring_score}")
    elif avg_words >= 25:
        archetypes[person] = ("THE STORYTELLER", f"avg {avg_words:.1f} words per msg")
    elif drama_pct >= 25:
        archetypes[person] = ("THE DRAMA QUEEN", f"{drama_pct:.1f}% ALL-CAPS msgs")
    elif silent_pct >= 50:
        archetypes[person] = ("THE GHOST", f"silent {streaks[person]} days")
    elif phil_score >= 5:
        archetypes[person] = ("THE LATE-NIGHT PHILOSOPHER", "deep thinker")
    else:
        archetypes[person] = ("THE QUESTION MASTER", "default/tiebreaker")

In [ ]:
print("================================================================")
print("                   THE GROUP CHAT EXPOSED                   ")
print("================================================================")

start = start_date.strftime('%d %B %Y')
end = end_date.strftime('%d %B %Y')
print(f"Chat Lifespan  : {start} to {end}")
print(f"Total Yapping  : {len(messages)} messages")
print(f"Most Chaotic   : {busiest_day[0]} ({busiest_day[1]} messages)")
print(f"Peak Yap Hour  : {busiest_hour[0]} ({busiest_hour[1]} messages)")
print("----------------------------------------------------------------")

print("\n :) WHO TALKS THE MOST?")
for p_info in sorted_participants:
    name = p_info[0]
    count = p_info[1]
    pct = (count / len(messages)) * 100
    display_name = name[:12] if len(name) > 12 else name
    print(f"  {display_name:<12} : {count:>5} messages ({pct:>4.1f}%)")

print("\n THE GROUP'S FAVORITE SLANG")
if len(top_words) > 0:
    max_count = top_words[0][1]
else:
    max_count = 1

for word_info in top_words:
    word = word_info[0]
    count = word_info[1]
    bar = "█" * int((count / max_count) * 15)
    print(f"  {word:<12} {count:<5} {bar}")

print("\n TERMINALLY ONLINE vs. THE GHOSTS")
valid_responses = []
for k in avg_responses:
    v = avg_responses[k]
    if v != 999999:
        valid_responses.append((k, v))

if len(valid_responses) > 0:
    fastest = min(valid_responses, key=get_count)
    slowest = max(valid_responses, key=get_count)

    fast_name = fastest[0][:12] if len(fastest[0]) > 12 else fastest[0]
    slow_name = slowest[0][:12] if len(slowest[0]) > 12 else slowest[0]

    fast_time = fastest[1] / 60
    slow_time = slowest[1] / 3600
    print(f"  Fastest texter : {fast_name} (replies in ~{fast_time:.1f} mins)")
    print(f"  Slowest texter : {slow_name} (takes ~{slow_time:.1f} hours to reply)")

print("\n LONGEST DISAPPEARING ACTS")
sorted_streaks = sorted(streaks.items(), key=get_count, reverse=True)
for streak_info in sorted_streaks:
    name = streak_info[0]
    days = streak_info[1]
    display_name = name[:12] if len(name) > 12 else name
    print(f"  {display_name:<12} : Ghosted for {days} days straight")

print("\n FINAL VERDICT: PERSONALITY ARCHETYPES")
for name in participant_names:
    arch_info = archetypes[name]
    arch = arch_info[0]
    reason = arch_info[1]
    display_name = name[:12] if len(name) > 12 else name
    print(f"  {display_name:<12} -> {arch:<25} ({reason})")

print("================================================================")

print("================================================================")